In [ ]:
# worked with Parker, Hansika, Chanchal, Naga

In [ ]:
import torch

In [ ]:
from google.colab import drive #gets drive
drive.mount('/content/drive') # links colab to drive
dataset_path = '/content/drive/MyDrive/Data_for_training_coneGPT' #tells code exactly what folder

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import torchvision.transforms as transform1 # this library helps us resize/crop image so the model can learn from it easier
transforms = transform1.Compose( # compose is a method that does multiple image transformations
 [
        transform1.Resize((128,128)), #resizes the image so it is easier to read
        transform1.ToTensor(), #converts NumPy array to tensor
        transform1.Normalize([0.5,0.5,0.5], [0.5,0.5,0.5]) # this scales pixel values, helps model, we use 0.5 because nueral networks shift from -1 to 1
  ]
)


In [ ]:
from torchvision.datasets import ImageFolder # this library loads images from folders, each subfolder is treated like a class
dataset = ImageFolder(root = dataset_path, transform = transforms) # loads all images from folder, transforms each image with the parameters we set in the prevoius cell,
print("Folder names:", dataset.classes) # folder names, each of the subfolders is considered a class
print("number of images:", len(dataset))

Folder names: ['Cone_images', 'No_cone_images']
number of images: 6


In [ ]:
from torch.utils.data import DataLoader
training = DataLoader(dataset, batch_size = 2, shuffle = True) # feeds model 2 images at a time, shuffles data so model can understand/train better, uses 'dataset' which we loaded in prevoius cell


In [ ]:
#now we have to build a CNN model which is Convolution Nueral Network, which is a nueral network specifically for images
# CNN looks for patterns(shapes,textures) using convolution layers(takes in small amount of data)
# automatically learn features
# CNN structure: convolutional layers + pooling + fully connected layers → output classes.
# pooling takes max/average in small patches, helps model focus on important stuff
#fully connected layers uses pooling to classify, which image belongs to which class

In [ ]:
import torch.nn as nn # imports nueral network model, which has layers(mentioned in last cell) and other functions
import torch.nn.functional as functional # imports pooling, dropout and other things needed to build nueral network
class coneGPT(nn.Module): #builidng a class called coneGPT, , inherits all important methods from nn.Module
  def __init__(self): #initialzes a layer, self means like coneGPT, refering to itself
    super(coneGPT, self).__init__()# super calls constructor of parent class(nn.module), this line makes sure that the model properly gets all features from parent class
    self.con1 = nn.Conv2d(3,10,3) # makes first convultion layer, self means it belongs to conGPT, nn.Conv2D means it is a convultion operator for 2D images
    #convultion layer is a filter that goes over image and tells us if there is a strong pattern on the image, creates feature maps
    #(3,16,3) in channels, out_channels_kernel size, 3 because 3 colors(red, blue, green), 10 because 10 feature maps/10 filters  , 3 becuase filter size is 3x3 which is common filter size
    self.pool = nn.MaxPool2d(2, 2)      #pooling layer, takes 2 x2 patches
    self.con2 = nn.Conv2d(10, 16, 3)     # another convolution layer, outputs 16 feature map

    # Calculate the output size of the last pooling layer dynamically
    def _get_conv_output_size(self, shape):
        batch_size = 1
        input = torch.autograd.Variable(torch.rand(batch_size, *shape))
        output_feat = self.pool(functional.relu(self.con1(input)))
        output_feat = self.pool(functional.relu(self.con2(output_feat)))
        return output_feat.size(1) * output_feat.size(2) * output_feat.size(3)

    self.conv_output_size = _get_conv_output_size(self, (3, 128, 128)) # Assuming input image size is 128x128 with 3 channels

    self.functionalc1 = nn.Linear(self.conv_output_size, 128) # fully connected dense layer, "flattens " output from convulution layers
    self.functionalc2 = nn.Linear(128, 2)        #final output layer,

  def forward(self, x): # so this class is to be used on each image
      x = self.pool(functional.relu(self.con1(x)))  #apply con1 onto input x(image)
      x = self.pool(functional.relu(self.con2(x))) #apply con2 onto input x(image)
      x = x.view(-1, self.conv_output_size)     #flatten tensor meaning the layers speak to eachother?
      x = functional.relu(self.functionalc1(x))
      x = self.functionalc2(x)
      return x # outputs prediction of whehter yes a cone or no

model = coneGPT()
print(model)

coneGPT(
  (con1): Conv2d(3, 10, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (con2): Conv2d(10, 16, kernel_size=(3, 3), stride=(1, 1))
  (functionalc1): Linear(in_features=14400, out_features=128, bias=True)
  (functionalc2): Linear(in_features=128, out_features=2, bias=True)
)


In [ ]:
import torch.optim as optim #optim is a library that is used to help model learn based on errors it makes
criterion = nn.CrossEntropyLoss() #measures how wrong model is, compares models outputs to true values
optimizer = optim.Adam(model.parameters(), lr=0.001) #model adjusts based on mistakes, this defines how it should update?lr is learning rate, adam is probably an optimizer

In [ ]:
# Check if GPU is available and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device) # Move the model to the selected device

number_of_trails = 5 # all images will pass through model 5 times
for epoch in range(number_of_trails): #epoch means trails, so we are making a loop for the number of trails
  running_loss = 0.0 # wont lose results after one trail
  for images, labels in training: #creates for loop for images
    images, labels = images.to(device), labels.to(device) #moves data from CPU to where model is

    optimizer.zero_grad()           # reset gradients
    outputs = model(images)         # forward pass
    loss = criterion(outputs, labels)  # calculate loss
    loss.backward()                 # backpropagation
    optimizer.step()                # update weights

    running_loss += loss.item()

print(f"Epoch {epoch+1}, Loss: {running_loss/len(training)}")
torch.save(model.state_dict(), '/content/drive/MyDrive/cone_model.pth')

Epoch 5, Loss: 0.49415403604507446
